In [ ]:
import pandas as pd
# Load the DataFrame from analysis.ipynb (it's saved as a pickle)
df_essen_only = pd.read_pickle('pickle_files/df_essen_only.pkl')

In [9]:
df_essen_only.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5724 entries, 5 to 13127
Data columns (total 31 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   date                       5724 non-null   datetime64[ns]
 1   original_name              5724 non-null   object        
 2   target_amount              5724 non-null   float64       
 3   sold_amount                5724 non-null   int64         
 4   dish_name                  5724 non-null   object        
 5   weekday                    5724 non-null   int32         
 6   year                       5724 non-null   int32         
 7   dish_type                  5724 non-null   object        
 8   month                      5724 non-null   int32         
 9   week                       5724 non-null   UInt32        
 10  day                        5724 non-null   int32         
 11  is_weekend                 5724 non-null   int32         
 12  weekofyear

In [10]:
df_essen_only.head()

,date,original_name,target_amount,sold_amount,dish_name,weekday,year,dish_type,month,week,...,rolling_7d_target_median,rolling_30d_sold_mean,rolling_30d_sold_median,rolling_30d_target_mean,rolling_30d_target_median,daily_total_sales,rolling_7d_total_sales,rolling_30d_total_sales,dish_freq,is_essen
5,2014-01-07,Essen 3,190.0,185,Pasta mit Tomate-Gemüsesoße,1,2014,Essen (Menu),1,2,...,66.616574,66.668705,62.374719,69.736775,65.690484,861,861.000000,861.000000,419,1
6,2014-01-07,Essen 2,240.0,237,Puten Cordon Bleu mit Zitrone,1,2014,Essen (Menu),1,2,...,66.616574,66.668705,62.374719,69.736775,65.690484,861,861.000000,861.000000,1304,1
7,2014-01-07,Essen 1,162.0,159,Hacklett - Hacksteak,1,2014,Essen (Menu),1,2,...,66.616574,66.668705,62.374719,69.736775,65.690484,861,861.000000,861.000000,1812,1
13,2014-01-08,Essen 4,60.0,52,Nudelpfanne mit Muschelfleisch und Papri,2,2014,Essen (Menu),1,2,...,66.616574,66.668705,62.374719,69.736775,65.690484,676,728.857143,789.846154,1523,1
14,2014-01-08,Essen 5,270.0,265,Rösti Hauptgericht,2,2014,Essen (Menu),1,2,...,66.616574,66.668705,62.374719,69.736775,65.690484,676,702.428571,781.714286,666,1


In [11]:
df_essen_only['dish_name'].nunique()

1035

In [ ]:
import google.generativeai as genai

# Configure the API
genai.configure(api_key="GEMINI_API_KEY")
model = genai.GenerativeModel('models/embedding-001')

# Get unique dish names
unique_dishes = df_essen_only['dish_name'].dropna().unique()

# Get embeddings
dish_embeddings = {}
for dish in unique_dishes:
    embedding = genai.embed_content(model = 'models/embedding-001',
                                    content=dish,
                                    task_type="SEMANTIC_SIMILARITY")['embedding']
    dish_embeddings[dish] = embedding

In [14]:
import pickle  

# Save embeddings to a pickle file
with open('pickle_files/dish_embeddings.pkl', 'wb') as f:
    pickle.dump(dish_embeddings, f)

In [18]:
embedding_df = pd.DataFrame.from_dict(dish_embeddings, orient='index')
embedding_df.reset_index(inplace=True)
embedding_df.rename(columns={'index': 'dish_name'}, inplace=True)
embedding_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1035 entries, 0 to 1034
Columns: 769 entries, dish_name to 767
dtypes: float64(768), object(1)
memory usage: 6.1+ MB


In [19]:
embedding_df.head()

,dish_name,0,1,2,3,4,5,6,7,8,...,758,759,760,761,762,763,764,765,766,767
0,Pasta mit Tomate-Gemüsesoße,0.021249,-0.034028,0.005527,0.009006,0.047246,0.023477,-0.002105,-0.016127,0.035557,...,-0.004310,0.020344,-0.051341,0.032908,-0.029573,0.000782,-0.043373,0.032240,-0.020990,-0.014158
1,Puten Cordon Bleu mit Zitrone,0.017317,-0.056851,-0.070699,-0.027099,0.014342,0.008165,-0.011565,-0.006481,0.034186,...,0.022824,0.018682,-0.067907,0.023231,-0.040663,0.060269,-0.005830,0.018221,-0.056192,-0.030810
2,Hacklett - Hacksteak,0.021459,-0.000691,-0.059774,-0.066763,0.038330,-0.001738,-0.013816,-0.016254,0.062202,...,0.019436,-0.001189,-0.023183,-0.011423,0.033921,0.018404,0.020297,-0.032669,-0.039641,0.008642
3,Nudelpfanne mit Muschelfleisch und Papri,-0.036713,-0.016585,0.000595,0.018914,0.016347,0.012880,-0.001823,-0.025439,0.029771,...,0.039184,0.022784,-0.046973,0.055940,-0.021821,-0.014707,-0.011800,0.017561,-0.032900,0.037525
4,Rösti Hauptgericht,-0.008220,-0.019783,-0.044453,-0.033724,0.014393,0.045740,-0.030042,-0.046713,0.018144,...,-0.017783,0.023003,-0.033578,0.006598,-0.025619,0.045689,-0.072074,-0.000586,-0.014357,0.013391
